In [ ]:
#import os
#os.environ["CUDA_DEVICE_ORDER"]="PCI_BUS_ID"
#os.environ["CUDA_VISIBLE_DEVICES"]="6"  # specify which GPU(s) to be used 6,7

In [1]:
import torch

#torch.cuda.is_available()
torch.cuda.get_device_name(0)

'NVIDIA GeForce RTX 2060'

# Load BERT model

In [2]:
from transformers import AutoModelForMaskedLM

model_checkpoint = "bert-base-uncased"
model = AutoModelForMaskedLM.from_pretrained(model_checkpoint)

/home/andres/Documentos/Big Data/TFM/adaptación de ezra/.venv_adap_ezra/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 202/202 [00:00<00:00, 5884.86it/s]
BertForMaskedLM LOAD REPORT from: bert-base-uncased
Key                         | Status     |  | 
----------------------------+------------+--+-
cls.seq_relationship.bias   | UNEXPECTED |  | 
bert.pooler.dense.bias      | UNEXPECTED |  | 
cls.seq_relationship.weight | UNEXPECTED |  | 
bert.pooler.dense.weight    | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [4]:
distilbert_num_parameters = model.num_parameters() / 1_000_000
print(f"'>>> BERT number of parameters: {round(distilbert_num_parameters)}M'")

'>>> BERT number of parameters: 110M'


## Tokenize text example

In [4]:
text = "This is a great [MASK]."

In [5]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

/home/aragon.ezra/anaconda3/envs/eriskLLM/lib/python3.9/site-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


In [6]:
import torch

inputs = tokenizer(text, return_tensors="pt")
token_logits = model(**inputs).logits
# Find the location of [MASK] and extract its logits
mask_token_index = torch.where(inputs["input_ids"] == tokenizer.mask_token_id)[1]
mask_token_logits = token_logits[0, mask_token_index, :]
# Pick the [MASK] candidates with the highest logits
top_5_tokens = torch.topk(mask_token_logits, 5, dim=1).indices[0].tolist()

for token in top_5_tokens:
    print(f"'>>> {text.replace(tokenizer.mask_token, tokenizer.decode([token]))}'")

2025-02-18 15:24:05.779543: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2025-02-18 15:24:05.779603: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2025-02-18 15:24:05.794828: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-02-18 15:24:05.838128: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-02-18 15:24:06.614513: W tensorflow/compiler/tf2

'>>> This is a great idea.'
'>>> This is a great day.'
'>>> This is a great place.'
'>>> This is a great time.'
'>>> This is a great thing.'


# Load dataset

In [7]:
import pandas as pd
from datasets import Dataset, load_dataset, concatenate_datasets

#Original
raw_dep_1 = pd.read_csv("selfharm/selfharm_1.csv", usecols = ['title'], encoding='unicode_escape')
raw_dep_1.rename(columns={"title": "post"}, inplace=True)
raw_dep_11 = pd.read_csv("selfharm/selfharm_1.csv",  usecols = ['body'], encoding='unicode_escape')
raw_dep_11.rename(columns={"body": "post"}, inplace=True)
dep_posts1 = raw_dep_1.dropna()
dep_posts11 = raw_dep_11.dropna()

raw_dep_2 = pd.read_csv("selfharm/selfharm_2.csv", usecols = ['title'], encoding='unicode_escape')
raw_dep_2.rename(columns={"title": "post"}, inplace=True)
raw_dep_22 = pd.read_csv("selfharm/selfharm_2.csv",  usecols = ['body'], encoding='unicode_escape')
raw_dep_22.rename(columns={"body": "post"}, inplace=True)
dep_posts2 = raw_dep_2.dropna()
dep_posts22 = raw_dep_22.dropna()

#### Augmented
column_names = ["post"]
aug_posts_raw1 = pd.read_csv("selfharm/selfharm_augmented.csv", names=column_names, skiprows=1)
aug_posts1 = aug_posts_raw1.dropna()

aug_posts_raw2 = pd.read_csv("selfharm/selfharm_augmented2.csv", names=column_names, skiprows=1)
aug_posts2 = aug_posts_raw2.dropna()

# Convert to dataset format
dataset1 = Dataset.from_pandas(dep_posts1)
dataset11 = Dataset.from_pandas(dep_posts11)
dataset2 = Dataset.from_pandas(dep_posts2)
dataset22 = Dataset.from_pandas(dep_posts22)

dataset3 = Dataset.from_pandas(aug_posts1)
dataset4 = Dataset.from_pandas(aug_posts2)

dataset = concatenate_datasets([dataset1, dataset11, dataset2, dataset22, dataset3, dataset4])
dataset

/home/aragon.ezra/anaconda3/envs/eriskLLM/lib/python3.9/site-packages/pyarrow/pandas_compat.py:373: FutureWarning: is_sparse is deprecated and will be removed in a future version. Check `isinstance(dtype, pd.SparseDtype)` instead.
  if _pandas_api.is_sparse(col):


Dataset({
    features: ['post', '__index_level_0__'],
    num_rows: 64746
})

In [ ]:
sample = dataset.shuffle(seed=42).select(range(3))

for row in sample:
    print(f"\n'>>> Post: {row['post']}'")


'>>> Post: Hi, I'm 16 years old and I've been self-harming for two years. it started when I was in an abusive relationship and I was too scared to get out. when things finally ended after 6 months, I was clean for almost a year. I'm currently in a very healthy relationship with my girlfriend, and I'm the happiest I've ever been with someone, we're about to hit 8 months and yesterday the streak broke. when I hurt myself, it's usually because of an argument with my parents and I feel the need to punish myself out of guilt, or if I've built anger I take it on myself. I know that if my parents find out that I'm just telling me that I'm being dramatic and I just want attention, but self-harm makes me feel much better and I really need it. I came here looking for tips because I want to improve before I graduate from high school so I can have a happy future with my girlfriend.'

'>>> Post: It's been a little over 20 days since my long distance partner left me for someone else at work and the

# Preprocessing the data
For both auto-regressive and masked language modeling, a common preprocessing step is to concatenate all the examples and then split the whole corpus into chunks of equal size. This is quite different from our usual approach, where we simply tokenize individual examples. Why concatenate everything together? The reason is that individual examples might get truncated if they’re too long, and that would result in losing information that might be useful for the language modeling task!

we’ll first tokenize our corpus as usual, but without setting the truncation=True option in our tokenizer. We’ll also grab the word IDs if they are available, as we will need them later on to do whole word masking. We’ll wrap this in a simple function, and while we’re at it we’ll remove the text and label columns since we don’t need them any longer:

In [ ]:
def tokenize_function(examples):
    #examples["body"].remove("None,")
    #print(type(examples["body"]))
    result = tokenizer(examples["post"])

    if tokenizer.is_fast:
        result["word_ids"] = [result.word_ids(i) for i in range(len(result["input_ids"]))]
    return result


# Use batched=True to activate fast multithreading!
tokenized_datasets = dataset.map(
    tokenize_function, batched=True, remove_columns=['post','__index_level_0__']
)
tokenized_datasets

Map:   0%|          | 0/64746 [00:00<?, ? examples/s]

Token indices sequence length is longer than the specified maximum sequence length for this model (849 > 512). Running this sequence through the model will result in indexing errors


Dataset({
    features: ['input_ids', 'token_type_ids', 'attention_mask', 'word_ids'],
    num_rows: 64746
})

In [10]:
tokenizer.model_max_length

512

To show how the concatenation works, let’s take a few reviews from our tokenized training set and print out the number of tokens per review:

In [11]:
chunk_size = 128

# Slicing produces a list of lists for each feature
tokenized_samples = tokenized_datasets[:3]

for idx, sample in enumerate(tokenized_samples["input_ids"]):
    print(f"'>>> Post {idx} length: {len(sample)}'")

'>>> Post 0 length: 13'
'>>> Post 1 length: 31'
'>>> Post 2 length: 38'


We can then concatenate all these examples with a simple dictionary comprehension, as follows:

In [12]:
concatenated_examples = {
    k: sum(tokenized_samples[k], []) for k in tokenized_samples.keys()
}
total_length = len(concatenated_examples["input_ids"])
print(f"'>>> Concatenated reviews length: {total_length}'")

'>>> Concatenated reviews length: 82'


So now let’s split the concatenated reviews into chunks of the size given by block_size. To do so, we iterate over the features in concatenated_examples and use a list comprehension to create slices of each feature. The result is a dictionary of chunks for each feature:

In [13]:
chunks = {
    k: [t[i : i + chunk_size] for i in range(0, total_length, chunk_size)]
    for k, t in concatenated_examples.items()
}

for chunk in chunks["input_ids"]:
    print(f"'>>> Chunk length: {len(chunk)}'")

'>>> Chunk length: 82'


As you can see in this example, the last chunk will generally be smaller than the maximum chunk size. There are two main strategies for dealing with this:

- Drop the last chunk if it’s smaller than chunk_size.
- Pad the last chunk until its length equals chunk_size.

We’ll take the first approach here, so let’s wrap all of the above logic in a single function that we can apply to our tokenized datasets:

In [14]:
def group_texts(examples):
    # Concatenate all texts
    concatenated_examples = {k: sum(examples[k], []) for k in examples.keys()}
    # Compute length of concatenated texts
    total_length = len(concatenated_examples[list(examples.keys())[0]])
    # We drop the last chunk if it's smaller than chunk_size
    total_length = (total_length // chunk_size) * chunk_size
    # Split by chunks of max_len
    result = {
        k: [t[i : i + chunk_size] for i in range(0, total_length, chunk_size)]
        for k, t in concatenated_examples.items()
    }
    # Create a new labels column
    result["labels"] = result["input_ids"].copy()
    return result

Note that in the last step of group_texts() we create a new labels column which is a copy of the input_ids one. As we’ll see shortly, that’s because in masked language modeling the objective is to predict randomly masked tokens in the input batch, and by creating a labels column we provide the ground truth for our language model to learn from.

Let’s now apply group_texts() to our tokenized datasets using our trusty Dataset.map() function:

In [15]:
lm_datasets = tokenized_datasets.map(group_texts, batched=True)
lm_datasets

Map:   0%|          | 0/64746 [00:00<?, ? examples/s]

Dataset({
    features: ['input_ids', 'token_type_ids', 'attention_mask', 'word_ids', 'labels'],
    num_rows: 56131
})

You can see that grouping and then chunking the texts has produced many more examples than our original 25,000 for the train and test splits. That’s because we now have examples involving contiguous tokens that span across multiple examples from the original corpus. You can see this explicitly by looking for the special [SEP] and [CLS] tokens in one of the chunks:

In [16]:
tokenizer.decode(lm_datasets[1]["input_ids"])

'abilify [SEP] [CLS] bid dosing of trintellix [SEP] [CLS] why do i get hallucinations 1h after taking my daily quetiapine? getting worse every day [SEP] [CLS] how much is too much? [SEP] [CLS] is this something i can talk to my new therapist about? [SEP] [CLS] diagnosticize me [SEP] [CLS] is it harmful to stay awake on seroquel xr? [SEP] [CLS] 20f - feel incredibly wound up and restless [SEP] [CLS] can you recommend resources to self - manage narcissism? [SEP] [CLS] ssri wirhdrawal and lsd personality changes... help please. [SEP] [CLS] feeling dys'

In this example you can see two overlapping movie reviews, one about a high school movie and the other about homelessness. Let’s also check out what the labels look like for masked language modeling:

In [17]:
tokenizer.decode(lm_datasets[1]["labels"])

'abilify [SEP] [CLS] bid dosing of trintellix [SEP] [CLS] why do i get hallucinations 1h after taking my daily quetiapine? getting worse every day [SEP] [CLS] how much is too much? [SEP] [CLS] is this something i can talk to my new therapist about? [SEP] [CLS] diagnosticize me [SEP] [CLS] is it harmful to stay awake on seroquel xr? [SEP] [CLS] 20f - feel incredibly wound up and restless [SEP] [CLS] can you recommend resources to self - manage narcissism? [SEP] [CLS] ssri wirhdrawal and lsd personality changes... help please. [SEP] [CLS] feeling dys'

As expected from our group_texts() function above, this looks identical to the decoded input_ids — but then how can our model possibly learn anything? We’re missing a key step: inserting [MASK] tokens at random positions in the inputs! Let’s see how we can do this on the fly during fine-tuning using a special data collator.

# Fine-tuning BERT with the Trainer API

Fine-tuning a masked language model is almost identical to fine-tuning a sequence classification model. The only difference is that we need a special data collator that can randomly mask some of the tokens in each batch of texts. Fortunately, 🤗 Transformers comes prepared with a dedicated DataCollatorForLanguageModeling for just this task. We just have to pass it the tokenizer and an mlm_probability argument that specifies what fraction of the tokens to mask. We’ll pick 15%, which is the amount used for BERT and a common choice in the literature:

In [18]:
from transformers import DataCollatorForLanguageModeling

data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm_probability=0.15)

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


To see how the random masking works, let’s feed a few examples to the data collator. Since it expects a list of dicts, where each dict represents a single chunk of contiguous text, we first iterate over the dataset before feeding the batch to the collator. We remove the "word_ids" key for this data collator as it does not expect it:

In [19]:
samples = [lm_datasets[i] for i in range(2)]
for sample in samples:
    _ = sample.pop("word_ids")

for chunk in data_collator(samples)["input_ids"]:
    print(f"\n'>>> {tokenizer.decode(chunk)}'")


'>>> [CLS] new providers wonat prescribe my medication. why? [SEP] [CLS] [ 21f ] i was prescribed risperidone, quetiapine and olanzapine. what [MASK] [MASK] a better alternative for these? [SEP] [CLS] m33lance am currently on 20 mg [MASK]citalopr [MASK] for gad and it killed my sex drive please suggest how can [MASK] reduce or [MASK] ssri sexual side effects? [SEP] [CLS] is this appropriate behaviour from a psychiatrist [MASK] [SEP] [CLS] disorder where you feel perpetually targeted [MASK] a real person [MASK] [SEP] [CLS] how long do thoughts to self destruct last after starting antidepressants [SEP] [CLS] outside [MASK] [SEP] [CLS]'

'>>> abili [MASK] [SEP] [CLS] bid dosing of trintelli [MASK] [SEP] [CLS] why do i get hallucinations 1h after taking my daily quetiapine [MASK] getting worse every day [SEP] [CLS] [MASK] much [MASK] too much? [SEP] [CLS] is this something i can talk to my new [MASK] about? [SEP] [CLS] diagnosticize me [SEP] [CLS] is it harmful to [MASK] awake on seroquel

Nice, it worked! We can see that the [MASK] token has been randomly inserted at various locations in our text. These will be the tokens which our model will have to predict during training — and the beauty of the data collator is that it will randomize the [MASK] insertion with every batch!

When training models for masked language modeling, one technique that can be used is to mask whole words together, not just individual tokens. This approach is called whole word masking. If we want to use whole word masking, we will need to build a data collator ourselves. A data collator is just a function that takes a list of samples and converts them into a batch, so let’s do this now! We’ll use the word IDs computed earlier to make a map between word indices and the corresponding tokens, then randomly decide which words to mask and apply that mask on the inputs. Note that the labels are all -100 except for the ones corresponding to mask words.

In [ ]:
import pandas as pd

df = pd.read_csv('lexicon_words.txt', sep=" ", header=None)
lexicon_words = df[0].values.tolist()
print(len(lexicon_words))
#lexicon_words = " ".join(lexicon_words)
#print(lexicon_words)
lexicon_words1 = tokenizer(lexicon_words)
#print(len(lexicon_words1["input_ids"]))
try:
    lexicon_words1["input_ids"].remove(int(101))
    lexicon_words1["input_ids"].remove(int(102))
except:
    pass
print(len(lexicon_words1["input_ids"]))

7545
7545


In [ ]:
import collections
import numpy as np

from transformers import default_data_collator

wwm_probability = 0.2


#def whole_word_masking_data_collator(features, lexicon):
def whole_word_masking_data_collator(features):
    df = pd.read_csv('lexicon_words.txt', sep=" ", header=None)
    lexicon_words = df[0].values.tolist()
    #print(len(lexicon_words))
    #lexicon_words = " ".join(lexicon_words)
    #print(lexicon_words)
    lexicon_words1 = tokenizer(lexicon_words)
    #print(len(lexicon_words1["input_ids"]))
    try:
        lexicon_words1["input_ids"].remove(int(101))
        lexicon_words1["input_ids"].remove(int(102))
    except:
        pass
    #print(len(lexicon_words1["input_ids"]))
    lexicon = lexicon_words1["input_ids"]
    #############################################################
    #print(features)
    for feature in features:
        word_ids = feature.pop("word_ids")
        #print(word_ids)

        # Create a map between words and corresponding token indices
        mapping = collections.defaultdict(list)
        current_word_index = -1
        current_word = None
        mask = []
        for idx, (word_id,inp) in enumerate(zip(word_ids,feature["input_ids"])):
            #print(idx,word_id,inp)
            if word_id is not None:
                if word_id != current_word:
                    current_word = word_id
                    current_word_index += 1
                    if inp in lexicon: #Mask words in the lexicon
                        mask.append(1) #
                    else:              #
                        mask.append(0) #
                mapping[current_word_index].append(idx)
        #print(mapping[current_word_index])

        # Randomly mask words
        mask_len = np.random.binomial(1, wwm_probability, (len(mapping),))
        #print(len(mask_len))
        input_ids = feature["input_ids"]
        #print(len(input_ids))
        labels = feature["labels"]
        #print(len(labels))
        new_labels = [-100] * len(labels)

        #Calculate the % difference for the masking
        mask = np.array(mask)
        len_count = np.count_nonzero(mask_len == 1)
        mask_count = np.count_nonzero(mask == 1)
        #print(len_count,mask_count)

        if len_count == 0:
            #mask_len = np.random.binomial(1, wwm_probability, (len(mapping),))
            #len_count = np.count_nonzero(mask_len == 1)
            len_count = 15
            print("len value = 0")
        else:
            pass

        x = len_count-mask_count
        #print(x)
        #print(len_count)
        xy = x/len_count
        #print(xy)
        yy = wwm_probability*xy
        #print((yy))

        #Random mask the rest of values
        mask_2 = np.random.binomial(1, yy, (len(mapping),))
        #print(len(mask),len(mask_2))
        new_mask = mask + mask_2
        new_mask = np.where(new_mask > 1, 1, new_mask)

        for word_id in np.where(new_mask)[0]:
            word_id = word_id.item()
            #print(word_id)
            for idx in mapping[word_id]:
                #print(idx)
                new_labels[idx] = labels[idx]
                input_ids[idx] = tokenizer.mask_token_id
                #print(input_ids[idx])
    #df.close()
    return default_data_collator(features)

Next, we can try it on the same samples as before:

In [22]:
samples = [lm_datasets[i] for i in range(2)]
batch = whole_word_masking_data_collator(samples)
#batch = whole_word_masking_data_collator(samples,lexicon_words1["input_ids"])

for chunk in batch["input_ids"]:
    print(f"\n'>>> {tokenizer.decode(chunk)}'")


'>>> [CLS] new providers wonat prescribe [MASK] medication. why? [SEP] [CLS] [ 21f ] i [MASK] prescribed risperidone, quetiapine [MASK] olanzapine. [MASK]'s [MASK] better alternative for these [MASK] [SEP] [CLS] [MASK] [MASK] [MASK] [MASK] currently [MASK] [MASK] mg escitalopram for gad and it killed my sex drive please suggest how can i [MASK] [MASK] [MASK] ssri sexual side effects? [SEP] [CLS] is [MASK] appropriate behaviour from [MASK] psychiatrist? [SEP] [CLS] [MASK] where [MASK] [MASK] perpetually targeted by [MASK] real person? [SEP] [CLS] how [MASK] do thoughts to [MASK] destruct last [MASK] starting antidepressants [SEP] [CLS] outside studying [SEP] [CLS]'

'>>> abilify [SEP] [CLS] bid dosing of trintellix [SEP] [CLS] why [MASK] [MASK] get hallucinations 1h after [MASK] my [MASK] quetiapine? [MASK] worse every [MASK] [SEP] [CLS] [MASK] much is too much? [SEP] [CLS] [MASK] this something i can talk to my new therapist about? [SEP] [CLS] diagnosticize me [SEP] [CLS] is it harmfu

Now that we have two data collators, the rest of the fine-tuning steps are standard. Training can take a while on Google Colab if you’re not lucky enough to score a mythical P100 GPU 😭, so we’ll first downsample the size of the training set to a few thousand examples. Don’t worry, we’ll still get a pretty decent language model! A quick way to downsample a dataset in 🤗 Datasets is via the Dataset.train_test_split() function

In [24]:
train_size = 50_000
test_size = int(0.1 * train_size)

downsampled_dataset = lm_datasets.train_test_split(
    train_size=train_size, test_size=test_size, seed=42
)
downsampled_dataset

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'word_ids', 'labels'],
        num_rows: 50000
    })
    test: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'word_ids', 'labels'],
        num_rows: 5000
    })
})

This has automatically created new train and test splits, with the training set size set to 10,000 examples and the validation set to 10% of that — feel free to increase this if you have a beefy GPU! The next thing we need to do is log in to the Hugging Face Hub. If you’re running this code in a notebook, you can do so with the following utility function:

In [25]:
#from huggingface_hub import notebook_login

#notebook_login()

In [26]:
#!python -m pip install huggingface_hub
#!apt install git-lfs

In [27]:
#!git config --global credential.helper store

Once we’re logged in, we can specify the arguments for the Trainer:

In [28]:
from transformers import TrainingArguments

batch_size = 16
# Show the training loss with every epoch
logging_steps = len(downsampled_dataset) // batch_size
print(len(downsampled_dataset))
model_name = model_checkpoint.split("/")[-1]

training_args = TrainingArguments(
    output_dir=f"{model_name}-finetuned-selfharm",
    overwrite_output_dir=True,
    evaluation_strategy="epoch",
    learning_rate=2e-5,
    weight_decay=0.01,
    per_device_train_batch_size=batch_size,
    per_device_eval_batch_size=batch_size,
    push_to_hub=False, ################## HUB
    fp16=True,
    logging_steps=50,
    remove_unused_columns=False,
)

2


/home/aragon.ezra/anaconda3/envs/eriskLLM/lib/python3.9/site-packages/transformers/training_args.py:1525: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


Here we tweaked a few of the default options, including logging_steps to ensure we track the training loss with each epoch. We’ve also used fp16=True to enable mixed-precision training, which gives us another boost in speed. By default, the Trainer will remove any columns that are not part of the model’s forward() method. This means that if you’re using the whole word masking collator, you’ll also need to set remove_unused_columns=False to ensure we don’t lose the word_ids column during training.

Note that you can specify the name of the repository you want to push to with the hub_model_id argument (in particular, you will have to use this argument to push to an organization). For instance, when we pushed the model to the huggingface-course organization, we added hub_model_id="huggingface-course/distilbert-finetuned-imdb" to TrainingArguments. By default, the repository used will be in your namespace and named after the output directory you set, so in our case it will be "lewtun/distilbert-finetuned-imdb".

We now have all the ingredients to instantiate the Trainer. Here we just use the standard data_collator, but you can try the whole word masking collator and compare the results as an exercise:

In [29]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=downsampled_dataset["train"],
    eval_dataset=downsampled_dataset["test"],
    data_collator=whole_word_masking_data_collator,
)

/home/aragon.ezra/anaconda3/envs/eriskLLM/lib/python3.9/site-packages/accelerate/accelerator.py:494: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = torch.cuda.amp.GradScaler(**kwargs)


Unlike other tasks like text classification or question answering where we’re given a labeled corpus to train on, with language modeling we don’t have any explicit labels. So how do we determine what makes a good language model? Like with the autocorrect feature in your phone, a good language model is one that assigns high probabilities to sentences that are grammatically correct, and low probabilities to nonsense sentences. To give you a better idea of what this looks like, you can find whole sets of “autocorrect fails” online, where the model in a person’s phone has produced some rather funny (and often inappropriate) completions!

Assuming our test set consists mostly of sentences that are grammatically correct, then one way to measure the quality of our language model is to calculate the probabilities it assigns to the next word in all the sentences of the test set. High probabilities indicates that the model indicates that the model is not “surprised” or “perplexed” by the unseen examples, and suggests it has learned the basic patterns of grammar in the language. There are various mathematical definitions of perplexity, but the one we’ll use defines it as the exponential of the cross-entropy loss. Thus, we can calculate the perplexity of our pretrained model by using the model.evaluate() method to compute the cross-entropy loss on the test set and then taking the exponential of the result:

In [30]:
import math
import torch
import gc

gc.collect()
torch.cuda.empty_cache()

eval_results = trainer.evaluate()
print(f">>> Perplexity: {math.exp(eval_results['eval_loss']):.2f}")

>>> Perplexity: 4.15


In [31]:
#torch.cuda.memory_summary(device=None, abbreviated=False)

A lower perplexity score means a better language model, and we can see here that our starting model has a somewhat large value. Let’s see if we can lower it by fine-tuning! To do that, we first run the training loop:

In [32]:
trainer.train()

Epoch,Training Loss,Validation Loss,Model Preparation Time
1,0.517700,0.502799,0.002100
2,0.493700,0.486927,0.002100
3,0.496700,0.487436,0.002100


TrainOutput(global_step=9375, training_loss=0.524126943766276, metrics={'train_runtime': 3089.7702, 'train_samples_per_second': 48.547, 'train_steps_per_second': 3.034, 'total_flos': 9870180480000000.0, 'train_loss': 0.524126943766276, 'epoch': 3.0})

and then compute the resulting perplexity on the test set as before:

In [33]:
eval_results = trainer.evaluate()
print(f">>> Perplexity: {math.exp(eval_results['eval_loss']):.2f}")

>>> Perplexity: 1.62


Once training is finished, we can push the model card with the training information to the Hub (the checkpoints are saved during training itself):

In [34]:
#!git config --global credential.helper store
#notebook_login()

In [35]:
#trainer.push_to_hub()
trainer.save_model(f"./{model_name}-finetuned-selfharm")

# Using our fine-tuned model

You can interact with your fine-tuned model either by using its widget on the Hub or locally with the pipeline from 🤗 Transformers. Let’s use the latter to download our model using the fill-mask pipeline:

In [36]:
from transformers import pipeline
from transformers import AutoModelForSequenceClassification

#state_dict = torch.load(f"./{model_name}-finetuned-redditEzra", map_location="cpu")

mask_filler = pipeline(
    "fill-mask", model=f"{model_name}-finetuned-selfharm/")


#new_model = AutoModelForSequenceClassification.from_pretrained(f"./{model_name}-finetuned-redditEzra")

/home/aragon.ezra/anaconda3/envs/eriskLLM/lib/python3.9/site-packages/torchvision/io/image.py:13: UserWarning: Failed to load image Python extension: '/home/aragon.ezra/anaconda3/envs/eriskLLM/lib/python3.9/site-packages/torchvision/image.so: undefined symbol: _ZN3c1017RegisterOperatorsD1Ev'If you don't plan on using image functionality from `torchvision.io`, you can ignore this warning. Otherwise, there might be something wrong with your environment. Did you have `libjpeg` or `libpng` installed before building `torchvision` from source?
  warn(
/home/aragon.ezra/anaconda3/envs/eriskLLM/lib/python3.9/site-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(
Hardware accelerator e.g. GP

We can then feed the pipeline our sample text of “This is a great [MASK]” and see what the top 5 predictions are:

In [37]:
text = "I used to be able to [MASK]."

preds = mask_filler(text)

for pred in preds:
    print(f">>> {pred['sequence']}")

>>> i used to be able to heal.
>>> i used to be able to talk.
>>> i used to be able to breathe.
>>> i used to be able to live.
>>> i used to be able to cry.
